In [13]:
import json
from pathlib import Path
import pandas as pd

RESULTS_DIR = Path('results_MLP2layer')
METHODS = ['Entropy', 'MC_Dropout', 'BNN', 'DRUE']

EXPERIMENTS = [
    'exp1_3T3_to_3T3_scaffold',
    'exp2_3T3_to_3T3_tanimoto',
    'exp3_3T3_to_HEK_scaffold',
    'exp4_3T3_to_HEK_tanimoto',
    'exp5_HEK_to_HEK_scaffold',
    'exp6_HEK_to_HEK_tanimoto',
    'exp7_HEK_to_3T3_scaffold',
    'exp8_HEK_to_3T3_tanimoto',
]

records = []
for exp in EXPERIMENTS:
    json_path = RESULTS_DIR / exp / 'ood_roc.json'
    if not json_path.exists():
        print(f'Missing: {json_path}')
        continue
    data = json.load(open(json_path))
    row = {'experiment': exp}
    for method in METHODS:
        if method in data:
            row[f'{method}_AUC']        = data[method]['auc']
            row[f'{method}_AVG_UE_ID']  = data[method]['avg_uncertainty_id']
            row[f'{method}_AVG_UE_OOD'] = data[method]['avg_uncertainty_ood']
        else:
            row[f'{method}_AUC']        = None
            row[f'{method}_AVG_UE_ID']  = None
            row[f'{method}_AVG_UE_OOD'] = None
    records.append(row)

df_flat = pd.DataFrame(records).set_index('experiment')
print(f'Loaded {len(df_flat)} experiments')

Loaded 8 experiments


In [14]:
# ── Table 1: AUC ─────────────────────────────────────────────────────────────
# Rows = experiments, Columns = 4 methods

df_auc = df_flat[[f'{m}_AUC' for m in METHODS]].copy()
df_auc.columns = METHODS

def highlight_best_auc(row):
    numeric = row.dropna()
    if numeric.empty:
        return ['' for _ in row]
    best = numeric.max()
    return ['font-weight: bold; background-color: #d4edda' if v == best else '' for v in row]

print('Table 1: OOD Detection AUC (higher = better, best per row in green)')
df_auc.style \
    .format('{:.4f}', na_rep='—') \
    .apply(highlight_best_auc, axis=1) \
    .set_caption('OOD Detection AUC')

Table 1: OOD Detection AUC (higher = better, best per row in green)


,Entropy,MC_Dropout,BNN,DRUE
experiment,,,,
exp1_3T3_to_3T3_scaffold,0.5196,0.5169,0.5153,0.5112
exp2_3T3_to_3T3_tanimoto,0.6266,0.6142,0.6237,0.6149
exp3_3T3_to_HEK_scaffold,0.5103,0.5080,0.5047,0.5169
exp4_3T3_to_HEK_tanimoto,0.5820,0.5675,0.5735,0.6080
exp5_HEK_to_HEK_scaffold,0.5074,0.5049,0.5062,0.5287
exp6_HEK_to_HEK_tanimoto,0.5721,0.5621,0.5611,0.6053
exp7_HEK_to_3T3_scaffold,0.5267,0.5275,0.5246,0.4985
exp8_HEK_to_3T3_tanimoto,0.5858,0.5810,0.5848,0.6059


In [15]:
# ── Table 2: AVG_UE_ID and AVG_UE_OOD ────────────────────────────────────────
# Columns level 0 = 4 methods
# Columns level 1 = AVG_UE_ID / AVG_UE_OOD
# Expected: OOD > ID for each method (model detects shift)

col_tuples = [(m, metric)
              for m in METHODS
              for metric in ['AVG_UE_ID', 'AVG_UE_OOD']]
df_ue = df_flat[[f'{m}_{k}' for m, k in col_tuples]].copy()
df_ue.columns = pd.MultiIndex.from_tuples(col_tuples)

def highlight_ood_gt_id(df):
    """Green if OOD > ID, red if OOD <= ID."""
    styles = pd.DataFrame('', index=df.index, columns=df.columns)
    for method in METHODS:
        id_col  = (method, 'AVG_UE_ID')
        ood_col = (method, 'AVG_UE_OOD')
        for exp in df.index:
            id_val  = df.loc[exp, id_col]
            ood_val = df.loc[exp, ood_col]
            if pd.isna(id_val) or pd.isna(ood_val):
                continue
            color = 'background-color: #d4edda' if ood_val > id_val else 'background-color: #f8d7da'
            styles.loc[exp, id_col]  = color
            styles.loc[exp, ood_col] = color
    return styles

print('Table 2: Average Uncertainty (green = OOD > ID ✓, red = OOD <= ID ✗)')
df_ue.style \
    .format('{:.6f}', na_rep='—') \
    .apply(highlight_ood_gt_id, axis=None) \
    .set_caption('Average Uncertainty: ID vs OOD')

Table 2: Average Uncertainty (green = OOD > ID ✓, red = OOD <= ID ✗)
